# Numerical evaluation the Finite massless double box Feynman integral

![alt text here](dbox.png)




This package is designed to evaluate Finite Feynman Integrals: the tensor representation and a numerical evaluation using Monte Carlo are the main objects produced. You must have an idea of the finiteness properties of the integral, e.g. using the methods of https://arxiv.org/pdf/2410.18014

In [ ]:
import sympy as sp

In [69]:
from feynite.lorentz import dL # Lorentz product
from feynite.constants import (pmom, kmom) # external momenta and loop momenta
from feynite.num_evaluator import pol_numerator # tensor representation
from feynite.monte_carlo_integration import  run_numerator # numerical integration

Define your kinematic data

In [70]:
# Define the lorentz vectors involved in your problem
k1, k2 = kmom(1), kmom(2)
p1, p2, p3, p4 = pmom(1), pmom(2), pmom(3), pmom(4)



Define the data of your graph

In [71]:
# symbols representing your external kinematic invariants
s12 = sp.Symbol('s12')
s23 = sp.Symbol('s23')
# rules satisfied by your external momenta
kin = {dL(p1, p1): 0, dL(p2, p2): 0, dL(p3, p3): 0, dL(p1, p2): s12/2, dL(p2, p3): s23/2,  dL(p1, p3):(-s12-s23)/2}

# The inverse propagators, masses, and loop momenta
dens = ([k1, k1-p1, k1-p1-p2, k1 - k2, k2-p1-p2, k2, k2-p1-p2-p3], [0, 0, 0, 0, 0, 0, 0], [k1, k2])

## The simplest case

In [72]:
# IN six dimensions the double box integral is finite
dim=6
parameters={s12: -7.0, s23:-1.0,  sp.Symbol('d'): float(dim)}

In [74]:
# The numerator should be of type sympy or a string in the variables defined above
# num="1" is also accepted
num=sp.S.One

In [75]:
res_integration=run_numerator(num, kin=kin, dens=dens, parameters=parameters, tag=None, outfile='mc_results.txt',  dim=dim, train=(25, 50_000), nitn=20 , neval=100_000, adapt=True, alpha=0.3, taming='tamed', sobol=None, workers=2, quiet=True, allow_divergent=True)

In [76]:
res_integration.keys()

dict_keys(['tag', 'rank', 'eU', 'eF', 'nvars', 'a', 'a_str', 'flag', 'corner_faces', 'value', 'sdev', 'Q', 'sobol', 'seconds', 'info', 'arrays', 'result', 'combined'])

In [78]:
#The result
res_integration[("result")]

-0.40588(51)

You can compare against a Scrambled Sobol sequences approach
using sobol=(m,r), where r scrambles with 2^m points per scramble
See next example

## A rank 2 finite numerator

In [79]:
import json # Read polynomials in a json file
# Dictionary to translate your files into readable sympy expression
ns = {
    "s12": s12, "s23": s23,
    "dL": dL,
    "k1": kmom(1),
    "k2": kmom(2),
    "p1": pmom(1),
    "p2": pmom(2),
    "p3": pmom(3)}

In [80]:
#numerators obtained followins the methods of https://arxiv.org/pdf/2410.18014
numrank = "2"
with open("double_box_rank" + numrank + "_numerators.json", "r") as f:
    expr_strings = json.load(f)
nums = [sp.sympify(s, locals=ns, evaluate=False) for s in expr_strings]


In [104]:
num=nums[0]

In [108]:
# Sparse expression
polynomial_num_sparse=pol_numerator(num,dens, kin)

In [113]:
#Non sparse expression: human readable
polynomial_num=sum(pol_numerator(num,dens, kin, sparse=False))
print(polynomial_num)

-d*s12**2*z(1)*z(3)*z(4)**2/4 - d*s12**2*z(1)*z(3)*z(4)*z(5)/4 - d*s12**2*z(1)*z(3)*z(4)*z(6)/4 - d*s12**2*z(1)*z(3)*z(4)*z(7)/4 - d*s12**2*z(1)*z(4)**2*z(5)/4 - d*s12**2*z(1)*z(4)*z(5)*z(6)/4 - d*s12**2*z(2)*z(4)*z(5)*z(6)/4 - d*s12**2*z(3)*z(4)**2*z(6)/4 - d*s12**2*z(3)*z(4)*z(5)*z(6)/4 - d*s12**2*z(4)**2*z(5)*z(6)/4 - d*s12*s23*z(1)*z(3)*z(4)**2/2 - d*s12*s23*z(1)*z(3)*z(4)*z(5)/2 - d*s12*s23*z(1)*z(3)*z(4)*z(6)/2 - d*s12*s23*z(1)*z(3)*z(4)*z(7)/2 - d*s12*s23*z(1)*z(4)**2*z(5)/2 - d*s12*s23*z(1)*z(4)*z(5)*z(6)/2 - 3*d*s12*s23*z(2)*z(4)**2*z(7)/4 - d*s12*s23*z(2)*z(4)*z(5)*z(6)/2 - d*s12*s23*z(3)*z(4)**2*z(6)/2 - d*s12*s23*z(3)*z(4)*z(5)*z(6)/2 - d*s12*s23*z(4)**2*z(5)*z(6)/2 - d*s23**2*z(2)*z(4)**2*z(7) + s12**2*z(1)*z(3)*z(4)**2/2 + s12**2*z(1)*z(3)*z(4)*z(5)/2 + s12**2*z(1)*z(3)*z(4)*z(6)/2 + s12**2*z(1)*z(3)*z(4)*z(7)/2 + s12**2*z(1)*z(4)**2*z(5)/2 + s12**2*z(1)*z(4)*z(5)*z(6)/2 + s12**2*z(2)*z(4)*z(5)*z(6)/2 + s12**2*z(3)*z(4)**2*z(6)/2 + s12**2*z(3)*z(4)*z(5)*z(6)/2 + s12**2*z(

In [81]:
dim=4
parameters={s12: -7.0, s23:-1.0,  sp.Symbol('d'): float(dim)}

In [95]:
res_integration=run_numerator(num, kin=kin, dens=dens, parameters=parameters, dim=dim, tag=None, outfile='mc_results-rank.txt',
                  train=(15, 50_000), nitn=10 , neval=200_000, adapt=False, alpha=0.3, taming='tamed', sobol=(19,10), workers=2, quiet=True)

In [96]:
res_integration["seconds"]

33.9533896446228

In [97]:
res_integration["result"]


0.8490(45)

In [98]:
res_integration["sobol"]


(0.8627640726556358, 0.012823926438100755)

In [86]:
res_integration["rank"]


2

In [87]:
res_integration["info"]

{'edges': 7,
 'loops': 2,
 'rank': 2,
 'dim': 4,
 'gauge_pos': 7,
 'nvars': 6,
 'num_zero': False}

A non-finite integral will print a warning with a audit for divergent faces

In [101]:
res_nonfinite=run_numerator(sp.S.One, kin=kin, dens=dens, parameters=parameters, dim=dim, tag=None, outfile='mc_results-rank.txt', train=(15, 50_000), nitn=10 , neval=200_000, adapt=True, alpha=0.3, taming='tamed', sobol=None, workers=1, quiet=True)

[mc_batch] WARNING None: z->0 corner audit FAILED on 8 of 63 faces (worst D_T+|T|-1 = -1 on T=(0, 1, 5)) -- the integral may be value-divergent there; RUNNING the MC anyway (results row flagged corner!).  If it truly diverges the diagnostics will show it: Q << 1, erratic estimates.


In [102]:
res_nonfinite["result"]

-2.53(13)e+05

In [103]:
res_nonfinite["Q"]


9.81661978425395e-10